# Lorenz (1963) — Deterministic Nonperiodic Flow

A three-variable ODE system exhibiting deterministic chaos, derived as a minimal model of atmospheric convection.

## Overview

The Lorenz (1963) system is a severely truncated model of Rayleigh–Bénard convection, retaining only three Fourier modes from a full 2-D fluid dynamics problem. Despite its simplicity the system produces bounded, non-periodic trajectories that are exponentially sensitive to initial conditions — the defining signature of deterministic chaos. It serves as the canonical demonstration that a deterministic system with three degrees of freedom can be unpredictable in practice, with immediate implications for weather forecasting and the limits of prediction in any chaotic physical system.

## Literature

Lorenz, E. N., 1963: Deterministic Nonperiodic Flow. J. Atmos. Sci., 20, 130–141, https://doi.org/10.1175/1520-0469_1963_020_0130_dnf_2_0_co_2.

**Figures reproduced below:**
- Figure 1 equivalent: irregular oscillations in a state variable over time
- Figure 2 equivalent: trajectory projection onto the y–z phase plane (the "butterfly")
- Supplementary: sensitive dependence on initial conditions (the central result of the paper, illustrated rather than plotted)

## Equations

$$
\frac{dx}{dt} = \sigma(y - x)
$$

$$
\frac{dy}{dt} = x(\rho - z) - y
$$

$$
\frac{dz}{dt} = xy - \beta z
$$

| Symbol | Meaning |
|--------|---------|
| $x$ | Rate of convective overturning |
| $y$ | Horizontal temperature contrast between ascending and descending fluid |
| $z$ | Deviation of the vertical temperature profile from linearity |
| $\sigma$ | Prandtl number (ratio of kinematic viscosity to thermal diffusivity) |
| $\rho$ | Normalized Rayleigh number; controls the intensity of convection |
| $\beta$ | Geometric factor; depends on the aspect ratio of the convection cells |

## Parameters

| Symbol | `param_values` key | Default | Physical meaning |
|--------|-------------------|---------|------------------|
| $\sigma$ | `sigma` | 10.0 | Prandtl number |
| $\rho$ | `rho` | 28.0 | Normalized Rayleigh number; chaos onset at $\rho \approx 24.7$ |
| $\beta$ | `beta` | 8/3 ≈ 2.667 | Geometric factor |

All three parameters support scalars, callables `(t)` / `(t, state)` / `(t, state, model)`, and `Forcing` objects.

## State variables

| Name | Integrated | Description |
|------|-----------|-------------|
| `x` | Yes | Convective overturning rate |
| `y` | Yes | Horizontal temperature contrast |
| `z` | Yes | Vertical temperature deviation from linearity |

## Implementation notes

`Lorenz63.dydt` appends to `state_variables` and `time` during integration rather than using the post-history hook. Use `model.reframe_time_axis(t_eval)` after integration to obtain output on a uniform time grid. `'RK45'` is the recommended solver; the system is smooth and continuous.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import paleobeasts as pb

## Setup

Canonical parameters from Lorenz (1963). The model is unforced. Initial conditions $(-8, 8, 27)$ place the trajectory close to but not on the attractor, giving a transient of roughly 2–3 time units before the irregular oscillations characteristic of the paper appear.

In [ ]:
sigma, rho, beta = 10.0, 28.0, 8/3

model = pb.Lorenz63(forcing=pb.Forcing(0.0), sigma=sigma, rho=rho, beta=beta)

t_span = (0, 50)
y0 = [-8.0, 8.0, 27.0]

model.integrate(t_span=t_span, y0=y0, method='RK45')
t_eval = pb.utils.define_t_eval(t_span, delta_t=0.01)
state = model.reframe_time_axis(t_eval)
t = model.time

## Reproducing Lorenz (1963) Figure 1: irregular oscillations in a state variable

Lorenz's Figure 1 shows the time evolution of $N$ (his notation for convective overturning, analogous to $y$ here), demonstrating that the trajectory is non-periodic: it oscillates irregularly between positive and negative excursions without ever exactly repeating. All three state variables are shown to make the non-periodicity visually unambiguous.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 5), sharex=True)
labels = ['$x$', '$y$', '$z$']
colors = ['#1f77b4', '#d62728', '#2ca02c']

for ax, var, label, color in zip(axes, ['x', 'y', 'z'], labels, colors):
    ax.plot(t, state[var], lw=0.6, color=color)
    ax.set_ylabel(label, fontsize=13, rotation=0, labelpad=12)
    ax.set_xlim(t_span)

axes[-1].set_xlabel('time (dimensionless)', fontsize=11)
plt.tight_layout()
plt.show()

## Reproducing Lorenz (1963) Figure 2: attractor projection onto the phase plane

Lorenz's Figure 2 shows the trajectory projected onto the $y$–$z$ plane, revealing a two-lobed structure later named the "butterfly attractor." The trajectory winds around one lobe for an unpredictable number of revolutions before switching to the other, never visiting the exact same point twice. The $x$–$z$ projection (right) is the more widely reproduced view and makes the bilateral symmetry of the attractor most visible. Colour encodes time, showing how the trajectory visits each lobe.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc0 = axes[0].scatter(state['y'], state['z'], c=t, cmap='plasma', s=0.3, lw=0)
plt.colorbar(sc0, ax=axes[0], label='time')
axes[0].set_xlabel('$y$', fontsize=13)
axes[0].set_ylabel('$z$', fontsize=13)
axes[0].set_title('y\u2013z projection (cf. Lorenz 1963 Fig. 2)', fontsize=11)

sc1 = axes[1].scatter(state['x'], state['z'], c=t, cmap='plasma', s=0.3, lw=0)
plt.colorbar(sc1, ax=axes[1], label='time')
axes[1].set_xlabel('$x$', fontsize=13)
axes[1].set_ylabel('$z$', fontsize=13)
axes[1].set_title('x\u2013z projection (the "butterfly")', fontsize=11)

plt.tight_layout()
plt.show()

## Sensitive dependence on initial conditions

The central result of Lorenz (1963) is that the system has a positive Lyapunov exponent: arbitrarily close initial conditions diverge exponentially. Two trajectories starting at distance $\varepsilon = 10^{-8}$ apart grow to $\mathcal{O}(1)$ separation within roughly 20 time units, after which all predictive information about one trajectory from knowledge of the other is lost. This is computed here by integrating a second model with $x_0$ perturbed by $\varepsilon$ and plotting the Euclidean separation on a log scale.

In [ ]:
epsilon = 1e-8

model_p = pb.Lorenz63(forcing=pb.Forcing(0.0), sigma=sigma, rho=rho, beta=beta)
model_p.integrate(t_span=t_span, y0=[-8.0 + epsilon, 8.0, 27.0], method='RK45')
state_p = model_p.reframe_time_axis(t_eval)

separation = np.sqrt(
    (state['x'] - state_p['x'])**2
    + (state['y'] - state_p['y'])**2
    + (state['z'] - state_p['z'])**2
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(t, separation + 1e-16, color='#333333', lw=1.0)
ax.axhline(1.0, color='#d62728', ls='--', lw=0.8, label='$\mathcal{O}(1)$ saturation')
ax.set_xlabel('time (dimensionless)', fontsize=11)
ax.set_ylabel(r'$|\Delta\mathbf{x}|$', fontsize=12)
ax.set_title(
    r'Sensitive dependence on initial conditions — exponential growth from $\varepsilon = 10^{-8}$',
    fontsize=11
)
ax.set_xlim(t_span)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Notes

- **Time units:** dimensionless; the original physical problem scales roughly as one overturning time of the convective fluid.
- **Chaos onset:** with $\sigma = 10$, $\beta = 8/3$, the system is chaotic for $\rho > 24.74$. At $\rho = 28$ (paper default) the system is well within the chaotic regime. For $\rho < 1$ the origin is the only fixed point; for $1 < \rho < 24.74$ there are two stable fixed points (convective rolls).
- **Forcing:** a scalar `Forcing` adds only to $dx/dt$; a 3-element vector `Forcing` adds component-wise to all three equations.
- **Related:** `lorenz96.ipynb` extends this structure to $n$ variables on a periodic ring.